# GAT Model Training Notebook

**CBSA – Continuous Behavioural Streaming API**

This notebook is fully self-contained:
1. Installs all required Python packages
2. Downloads training data from Azure Blob Storage (or generates synthetic data as fallback)
3. Trains the Graph Attention Network (GAT) with **CUDA** when available, otherwise **CPU**
4. Uploads the trained `.pt` checkpoint back to Azure Blob Storage

> **Configuration**: Set `AZURE_STORAGE_CONNECTION_STRING` (environment variable or in Cell 2)  
> to connect to your Azure storage account.


In [ ]:
# ── Install Required Packages ──────────────────────────────────────────────
import subprocess, sys

def _install(pkg: str) -> None:
    print(f"  pip install {pkg} ...", end=" ", flush=True)
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True,
    )
    print("ok" if r.returncode == 0 else f"FAILED\n{r.stderr[:300]}")

print("=" * 60)
print("Installing dependencies")
print("=" * 60)
_install("torch")
_install("torch-geometric")
_install("azure-storage-blob")
_install("numpy")
_install("tqdm")
print("\n✓  All packages processed")


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
import os

# ── Azure Blob Storage ──────────────────────────────────────────────────────
# Set AZURE_STORAGE_CONNECTION_STRING in your environment, or paste it below.
AZURE_CONNECTION_STRING: str = os.environ.get(
    "AZURE_STORAGE_CONNECTION_STRING", ""
)
AZURE_CONTAINER_NAME: str = os.environ.get(
    "AZURE_STORAGE_CONTAINER", "cbsa-models"
)
# Blob name for an optional training-data JSON stored in Azure
TRAINING_DATA_BLOB: str = "training_data.json"
# Blob name under which the trained checkpoint will be stored
CHECKPOINT_BLOB:    str = "gat_checkpoint.pt"
# Local path for the saved checkpoint
CHECKPOINT_PATH:    str = "gat_checkpoint.pt"

# ── Model Hyperparameters ────────────────────────────────────────────────────
INPUT_DIM    = 56    # 48 behavioural features + 8 event-type embedding
HIDDEN_DIM   = 64
OUTPUT_DIM   = 64
NUM_HEADS    = 4
DROPOUT      = 0.1
TEMPORAL_DIM = 16

# ── Training ─────────────────────────────────────────────────────────────────
NUM_EPOCHS     = 30
LEARNING_RATE  = 1e-3
TRIPLET_MARGIN = 0.5
RANDOM_SEED    = 42

# ── Data Windowing ────────────────────────────────────────────────────────────
WINDOW_SECONDS        = 20.0
WINDOW_STRIDE_SECONDS = 2.0
MIN_EVENTS_PER_WINDOW = 5
MAX_EVENTS_PER_WINDOW = 100

# ── Synthetic data fallback ───────────────────────────────────────────────────
SYNTH_NUM_USERS         = 5
SYNTH_SESSIONS_PER_USER = 12
SYNTH_EVENTS_PER_SESSION = 25

print("Configuration")
print(f"  Azure enabled  : {bool(AZURE_CONNECTION_STRING)}")
print(f"  Container      : {AZURE_CONTAINER_NAME}")
print(f"  Checkpoint blob: {CHECKPOINT_BLOB}")
print(f"  Epochs / LR / Margin : {NUM_EPOCHS} / {LEARNING_RATE} / {TRIPLET_MARGIN}")
print(f"  Model dims     : in={INPUT_DIM}  hidden={HIDDEN_DIM}  out={OUTPUT_DIM}  heads={NUM_HEADS}")


In [ ]:
# ── Imports & Device Setup ──────────────────────────────────────────────────
import json
import hashlib
import random
import time
import tempfile
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ── PyTorch Geometric ────────────────────────────────────────────────────────
try:
    from torch_geometric.nn import GATConv, global_mean_pool
    _TORCH_GEOMETRIC_OK = True
    print("✓ PyTorch Geometric available")
except ImportError as _pyg_err:
    _TORCH_GEOMETRIC_OK = False
    print(f"✗ PyTorch Geometric NOT available: {_pyg_err}")
    print("  Re-run Cell 2 or: pip install torch-geometric")

# ── Device selection: CUDA → MPS → CPU ───────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props  = torch.cuda.get_device_properties(0)
    print(f"✓ CUDA – {props.name}  ({props.total_memory // 1_048_576} MiB)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✓ Apple MPS available")
else:
    DEVICE = torch.device("cpu")
    print("i No GPU detected – training on CPU (fully supported, slower)")

print(f"\nActive device : {DEVICE}")
print(f"PyTorch       : {torch.__version__}")


In [ ]:
# ── Azure Blob Storage Helper ────────────────────────────────────────────────

class AzureBlobHelper:
    '''
    Self-contained Azure Blob Storage helper.

    Gracefully degrades when:
    * azure-storage-blob is not installed
    * connection_string is empty or invalid
    In both cases upload / download are no-ops and enabled == False.
    '''

    def __init__(self, connection_string: str, container_name: str) -> None:
        self._conn_str  = connection_string
        self._container = container_name
        self._client    = None
        self._enabled   = False
        if connection_string:
            self._connect()

    def _connect(self) -> None:
        try:
            from azure.storage.blob import BlobServiceClient
        except ImportError:
            print("azure-storage-blob not installed – blob operations disabled")
            return
        try:
            svc = BlobServiceClient.from_connection_string(self._conn_str)
            self._client = svc.get_container_client(self._container)
            try:
                self._client.create_container()
            except Exception:
                pass  # container already exists
            self._enabled = True
            print(f"✓ Azure Blob connected – container: {self._container}")
        except Exception as exc:
            print(f"Azure connection failed: {exc}")

    @property
    def enabled(self) -> bool:
        return self._enabled

    def list_blobs(self) -> List[str]:
        if not self._enabled:
            return []
        try:
            return [b.name for b in self._client.list_blobs()]
        except Exception:
            return []

    def download(self, blob_name: str, local_path: str) -> bool:
        '''Download blob_name to local_path. Returns True on success.'''
        if not self._enabled:
            return False
        try:
            os.makedirs(os.path.dirname(os.path.abspath(local_path)), exist_ok=True)
            blob = self._client.get_blob_client(blob_name)
            with open(local_path, "wb") as fh:
                fh.write(blob.download_blob().readall())
            print(f"✓ Downloaded  {blob_name}  ->  {local_path}")
            return True
        except Exception as exc:
            print(f"Download {blob_name} failed: {exc}")
            return False

    def upload(self, local_path: str, blob_name: str) -> bool:
        '''Upload local_path as blob_name. Returns True on success.'''
        if not self._enabled:
            return False
        try:
            blob = self._client.get_blob_client(blob_name)
            with open(local_path, "rb") as fh:
                blob.upload_blob(fh, overwrite=True)
            print(f"✓ Uploaded    {local_path}  ->  blob '{blob_name}'")
            return True
        except Exception as exc:
            print(f"Upload {blob_name} failed: {exc}")
            return False


blob = AzureBlobHelper(AZURE_CONNECTION_STRING, AZURE_CONTAINER_NAME)

print(f"\nBlob helper : {'ENABLED' if blob.enabled else 'disabled (no connection string)'}")
if blob.enabled:
    print(f"  Existing blobs: {blob.list_blobs() or '(empty container)'}")


In [ ]:
# ── Data Download & Synthetic Generation ────────────────────────────────────

def _et_embedding(event_type: str) -> List[float]:
    '''Deterministic 8-D embedding derived from SHA-256 of the event-type string.'''
    d = hashlib.sha256(str(event_type).encode()).digest()
    return [b / 255.0 for b in d[:8]]


def generate_synthetic_sessions(
    num_users:          int = SYNTH_NUM_USERS,
    sessions_per_user:  int = SYNTH_SESSIONS_PER_USER,
    events_per_session: int = SYNTH_EVENTS_PER_SESSION,
    seed:               int = RANDOM_SEED,
) -> List[Dict[str, Any]]:
    '''
    Generate synthetic behavioural sessions.

    Each user has a distinct 48-D behavioural centroid with small per-event
    Gaussian noise, simulating real intra-user consistency vs inter-user
    difference. Returns a list of session dicts matching fast_dataset.json schema.
    '''
    rng = np.random.RandomState(seed)
    EVENT_TYPES = [
        "PAGE_ENTER_HOME", "PAGE_EXIT_HOME", "CLICK_BUTTON",
        "SCROLL_DOWN",     "SCROLL_UP",      "KEY_PRESS",
        "FORM_SUBMIT",     "PAGE_ENTER_PROFILE", "PAGE_EXIT_PROFILE",
        "SWIPE_LEFT",      "SWIPE_RIGHT",    "TAP",
        "LONG_PRESS",      "PINCH_ZOOM",     "ROTATE",
    ]

    sessions = []
    for u_idx in range(num_users):
        uid      = f"user_{u_idx:03d}"
        centroid = rng.uniform(0.05, 0.95, 48)   # user-specific signature

        for s_idx in range(sessions_per_user):
            base_ts = 1_700_000_000.0 + u_idx * 10_000 + s_idx * 200
            ts      = base_ts
            events  = []

            for e_idx in range(events_per_session):
                ts   += rng.uniform(0.3, 4.0)
                etype = str(rng.choice(EVENT_TYPES))
                vec   = np.clip(centroid + rng.normal(0, 0.04, 48), 0.0, 1.0).tolist()
                events.append({
                    "timestamp":  round(ts, 3),
                    "event_type": etype,
                    "event_data": {
                        "timestamp": int(ts * 1_000),
                        "nonce":     f"{u_idx:02x}{s_idx:02x}{e_idx:02x}",
                        "vector":    vec,
                        "signature": hashlib.sha256(
                            f"{uid}|{s_idx}|{e_idx}".encode()
                        ).hexdigest(),
                    },
                })

            sessions.append({
                "session_id":   f"sess_{uid}_{s_idx:03d}",
                "user_id":      uid,
                "window_start": base_ts,
                "window_end":   ts,
                "events":       events,
            })

    return sessions


def load_sessions() -> List[Dict[str, Any]]:
    '''
    Load training sessions in priority order:
      1. Azure blob storage  (downloads TRAINING_DATA_BLOB)
      2. Local datasets/fast_dataset.json
      3. Synthetically generated data
    Returns a list of session dicts with keys: session_id, user_id, events, ...
    '''
    # ── 1. Azure ─────────────────────────────────────────────────────────────
    if blob.enabled and TRAINING_DATA_BLOB:
        tmp = os.path.join(tempfile.gettempdir(), "cbsa_training_data.json")
        if blob.download(TRAINING_DATA_BLOB, tmp):
            try:
                with open(tmp) as fh:
                    data = json.load(fh)
                sessions = data.get("sessions", [])
                users    = {s["user_id"] for s in sessions}
                if len(users) >= 2:
                    print(f"✓ Loaded {len(sessions)} sessions / {len(users)} users from Azure")
                    return sessions
                print(f"Azure data has only {len(users)} user(s) - need >= 2; falling back")
            except Exception as exc:
                print(f"Failed to parse Azure training data: {exc}")

    # ── 2. Local fast_dataset.json ────────────────────────────────────────────
    for candidate in [
        Path("datasets/fast_dataset.json"),
        Path("../datasets/fast_dataset.json"),
    ]:
        if candidate.exists():
            with open(candidate) as fh:
                data = json.load(fh)
            sessions = data.get("sessions", [])
            users    = {s["user_id"] for s in sessions}
            if len(users) >= 2:
                print(f"✓ Loaded {len(sessions)} local sessions / {len(users)} users")
                return sessions
            print(f"Local dataset has only {len(users)} user(s) - generating synthetic data")

    # ── 3. Synthetic ──────────────────────────────────────────────────────────
    print("No suitable real data found - generating synthetic behavioural data")
    sessions = generate_synthetic_sessions()
    users    = {s["user_id"] for s in sessions}
    print(f"✓ Generated {len(sessions)} synthetic sessions / {len(users)} users")
    return sessions


# ── Load ──────────────────────────────────────────────────────────────────────
all_sessions = load_sessions()
user_ids     = sorted({s["user_id"] for s in all_sessions})

print(f"\nDataset summary")
print(f"  Sessions : {len(all_sessions)}")
print(f"  Users    : {len(user_ids)}")
for uid in user_ids:
    cnt = sum(1 for s in all_sessions if s["user_id"] == uid)
    print(f"    {uid}: {cnt} sessions")


In [ ]:
# ── Data Models (standalone dataclasses, no pydantic required) ─────────────

@dataclass
class EventNode:
    node_id:           int
    timestamp:         float
    event_type:        str
    behavioral_vector: List[float]   # 56-D
    signature:         str = ""
    nonce:             str = ""


@dataclass
class TemporalEdge:
    source_node_id:  int
    target_node_id:  int
    time_delta:      float
    event_transition: str
    attention_weight: Optional[float] = None


@dataclass
class TemporalGraph:
    session_id:              str
    user_id:                 Optional[str]
    nodes:                   List[EventNode]
    edges:                   List[TemporalEdge]
    window_start:            float
    window_end:              float
    total_events:            int
    session_duration:        float
    event_diversity:         int
    avg_time_between_events: float


print("✓ Data models defined: EventNode, TemporalEdge, TemporalGraph")


In [ ]:
# ── Data Processor ──────────────────────────────────────────────────────────

class BehavioralDataProcessor:
    '''
    Converts a list of raw event dicts into a TemporalGraph.

    Expected event dict keys:
      timestamp        (float, Unix seconds)
      event_type       (str)
      event_data.vector (List[float], 48-D behavioural feature vector)
    '''

    def __init__(self, config: Optional[Dict[str, Any]] = None) -> None:
        cfg                           = config or {}
        self.max_events               = cfg.get("max_events_per_window",    MAX_EVENTS_PER_WINDOW)
        self.min_events               = cfg.get("min_events_per_window",    MIN_EVENTS_PER_WINDOW)
        self.distinct_event_limit     = cfg.get("distinct_event_connections", 4)

    @staticmethod
    def _et_embedding(event_type: str) -> List[float]:
        d = hashlib.sha256(str(event_type).encode()).digest()
        return [b / 255.0 for b in d[:8]]

    def _extract_vector(self, event: dict) -> List[float]:
        edata = event.get("event_data") or {}
        base  = list(edata.get("vector") or [])
        base  = (base + [0.0] * 48)[:48]
        none_count = sum(1 for v in base if v is None)
        if none_count:
            import warnings
            warnings.warn(
                f"event_data.vector contains {none_count} None value(s) in event "
                f"'{event.get('event_type', 'unknown')}' at t={event.get('timestamp')}; "
                "substituting 0.0",
                stacklevel=2,
            )
        base  = [float(v) if v is not None else 0.0 for v in base]
        emb   = self._et_embedding(event.get("event_type", "unknown"))
        return (base + emb)[:56]

    def process(
        self,
        events:     List[dict],
        user_id:    str,
        session_id: str,
    ) -> TemporalGraph:
        events = sorted(events, key=lambda e: e.get("timestamp", 0))
        events = events[-self.max_events:]

        nodes = [
            EventNode(
                node_id=i,
                timestamp=float(ev.get("timestamp", 0)),
                event_type=str(ev.get("event_type", "unknown")),
                behavioral_vector=self._extract_vector(ev),
                signature=str(ev.get("event_data", {}).get("signature", "")),
                nonce=str(ev.get("event_data", {}).get("nonce", "")),
            )
            for i, ev in enumerate(events)
        ]

        edges = self._build_edges(nodes)

        ts     = [n.timestamp for n in nodes]
        t0     = min(ts) if ts else 0.0
        t1     = max(ts) if ts else 0.0
        dur    = t1 - t0
        dts    = [ts[i + 1] - ts[i] for i in range(len(ts) - 1)]
        avg_dt = float(np.mean(dts)) if dts else 0.0

        return TemporalGraph(
            session_id=session_id,
            user_id=user_id,
            nodes=nodes,
            edges=edges,
            window_start=t0,
            window_end=t1,
            total_events=len(nodes),
            session_duration=dur,
            event_diversity=len({n.event_type for n in nodes}),
            avg_time_between_events=avg_dt,
        )

    def _build_edges(self, nodes: List[EventNode]) -> List[TemporalEdge]:
        edges = []
        limit = max(self.distinct_event_limit, 1)
        for i, src in enumerate(nodes):
            seen = {src.event_type}
            for j in range(i + 1, len(nodes)):
                tgt = nodes[j]
                edges.append(TemporalEdge(
                    source_node_id=src.node_id,
                    target_node_id=tgt.node_id,
                    time_delta=tgt.timestamp - src.timestamp,
                    event_transition=f"{src.event_type}->{tgt.event_type}",
                ))
                if tgt.event_type not in seen:
                    seen.add(tgt.event_type)
                    if len(seen) >= limit:
                        break
        return edges


class PyTorchDataConverter:
    '''Convert a TemporalGraph to PyTorch tensors (CPU; moved to DEVICE at training time).'''

    def convert(self, tg: TemporalGraph) -> SimpleNamespace:
        '''Returns a SimpleNamespace(x, edge_index, temporal_features, batch).'''
        nodes = tg.nodes
        edges = tg.edges

        x  = torch.FloatTensor([n.behavioral_vector for n in nodes])  # [N, 56]
        tf = torch.FloatTensor([[n.timestamp - tg.window_start] for n in nodes])  # [N, 1]

        if edges:
            src        = [e.source_node_id for e in edges]
            dst        = [e.target_node_id for e in edges]
            edge_index = torch.LongTensor([src, dst])
        else:
            idx        = list(range(len(nodes)))
            edge_index = torch.LongTensor([idx, idx])

        return SimpleNamespace(
            x=x,
            edge_index=edge_index,
            temporal_features=tf,
            batch=None,
        )


processor = BehavioralDataProcessor()
converter = PyTorchDataConverter()

print("✓ BehavioralDataProcessor and PyTorchDataConverter ready")


In [ ]:
# ── GAT Network (self-contained, no app.* imports) ──────────────────────────

if not _TORCH_GEOMETRIC_OK:
    raise RuntimeError(
        "PyTorch Geometric is required.\n"
        "Re-run Cell 2 or install manually: pip install torch-geometric"
    )


class TemporalGraphAttention(nn.Module):
    '''
    Two-layer multi-head Graph Attention Network with temporal encoding.

    Architecture
    ------------
    temporal_encoder  Linear(1 -> temporal_dim) + ReLU + Dropout
    gat_layer1        GATConv(input_dim + temporal_dim -> hidden_dim, heads)
                      LayerNorm + ReLU + Dropout
    gat_layer2        GATConv(hidden_dim -> output_dim, heads)
                      LayerNorm + ReLU
    global_pool       mean-pool over nodes, then FC(output_dim -> output_dim)
    '''

    def __init__(
        self,
        input_dim:    int   = INPUT_DIM,
        hidden_dim:   int   = HIDDEN_DIM,
        output_dim:   int   = OUTPUT_DIM,
        num_heads:    int   = NUM_HEADS,
        dropout:      float = DROPOUT,
        temporal_dim: int   = TEMPORAL_DIM,
    ) -> None:
        super().__init__()
        self.temporal_encoder = nn.Sequential(
            nn.Linear(1, temporal_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        combined   = input_dim + temporal_dim
        self.gat1  = GATConv(combined,   hidden_dim // num_heads,
                             heads=num_heads, dropout=dropout, concat=True)
        self.gat2  = GATConv(hidden_dim, output_dim // num_heads,
                             heads=num_heads, dropout=dropout, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.pool_fc = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
        self.drop = nn.Dropout(dropout)

    def forward(
        self,
        x:                 torch.Tensor,
        edge_index:        torch.Tensor,
        temporal_features: torch.Tensor,
        batch:             Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        '''Forward pass; returns graph-level embedding of shape [output_dim].'''
        t_enc = self.temporal_encoder(temporal_features)
        h     = torch.cat([x, t_enc], dim=1)

        h = self.gat1(h, edge_index)
        h = self.norm1(h)
        h = F.relu(h)
        h = self.drop(h)

        h = self.gat2(h, edge_index)
        h = self.norm2(h)
        h = F.relu(h)

        if batch is not None:
            g = global_mean_pool(h, batch)
        else:
            g = h.mean(dim=0, keepdim=True)

        return self.pool_fc(g).squeeze(0)   # [output_dim]


class SiameseGATNetwork(nn.Module):
    '''Wraps TemporalGraphAttention for pairwise / triplet metric learning.'''

    def __init__(self, config: Optional[Dict[str, Any]] = None) -> None:
        super().__init__()
        cfg = config or {}
        self.gat = TemporalGraphAttention(
            input_dim=cfg.get("input_dim",    INPUT_DIM),
            hidden_dim=cfg.get("hidden_dim",   HIDDEN_DIM),
            output_dim=cfg.get("output_dim",   OUTPUT_DIM),
            num_heads=cfg.get("num_heads",    NUM_HEADS),
            dropout=cfg.get("dropout",      DROPOUT),
            temporal_dim=cfg.get("temporal_dim", TEMPORAL_DIM),
        )

    def forward_once(self, data: SimpleNamespace) -> torch.Tensor:
        '''Single graph forward pass; tensors are moved to DEVICE automatically.'''
        return self.gat(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.temporal_features.to(DEVICE),
            data.batch,
        )

    def forward(
        self,
        data1: SimpleNamespace,
        data2: SimpleNamespace,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        '''Returns (cosine_similarity, emb1, emb2).'''
        emb1 = self.forward_once(data1)
        emb2 = self.forward_once(data2)
        sim  = F.cosine_similarity(emb1, emb2, dim=-1)
        return sim, emb1, emb2


class GATTrainer:
    '''Adam optimiser with cosine-similarity triplet loss and StepLR scheduling.'''

    def __init__(self, model: SiameseGATNetwork) -> None:
        self.model = model.to(DEVICE)
        self.opt   = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        self.sched = torch.optim.lr_scheduler.StepLR(
            self.opt, step_size=10, gamma=0.9
        )

    @staticmethod
    def _triplet_loss(
        anchor:   torch.Tensor,
        positive: torch.Tensor,
        negative: torch.Tensor,
    ) -> torch.Tensor:
        pos_sim = F.cosine_similarity(anchor, positive, dim=-1)
        neg_sim = F.cosine_similarity(anchor, negative, dim=-1)
        return torch.clamp(neg_sim - pos_sim + TRIPLET_MARGIN, min=0.0).mean()

    def step(
        self,
        anchor_data:   SimpleNamespace,
        positive_data: SimpleNamespace,
        negative_data: SimpleNamespace,
    ) -> float:
        '''One gradient-update step on a single triplet. Returns scalar loss.'''
        self.model.train()
        self.opt.zero_grad()
        a    = self.model.forward_once(anchor_data)
        p    = self.model.forward_once(positive_data)
        n    = self.model.forward_once(negative_data)
        loss = self._triplet_loss(a, p, n)
        loss.backward()
        self.opt.step()
        return float(loss.item())


n_params = sum(p.numel() for p in SiameseGATNetwork().parameters())
print(f"✓ TemporalGraphAttention, SiameseGATNetwork, GATTrainer defined")
print(f"  Total parameters in SiameseGATNetwork: {n_params:,}")


In [ ]:
# ── Prepare Training Graphs ─────────────────────────────────────────────────

def split_into_windows(
    events:     List[dict],
    window_sec: float = WINDOW_SECONDS,
    stride_sec: float = WINDOW_STRIDE_SECONDS,
    min_events: int   = MIN_EVENTS_PER_WINDOW,
) -> List[List[dict]]:
    '''Slide a fixed-size time window over a sorted event list.'''
    if not events:
        return []
    events  = sorted(events, key=lambda e: float(e.get("timestamp", 0)))
    t_first = events[0]["timestamp"]
    t_last  = events[-1]["timestamp"]
    windows, cur = [], t_first
    while cur <= t_last:
        w = [e for e in events if cur <= float(e.get("timestamp", 0)) < cur + window_sec]
        if len(w) >= min_events:
            windows.append(w)
        cur += stride_sec
    return windows


# ── Build per-user graph lists ────────────────────────────────────────────────
user_graphs: Dict[str, List[SimpleNamespace]] = {}

print("Converting sessions to graphs ...")
for uid in user_ids:
    uid_sessions = [s for s in all_sessions if s["user_id"] == uid]

    # Flatten events for this user then re-window
    all_events: List[dict] = []
    for sess in uid_sessions:
        all_events.extend(sess.get("events", []))

    windows = split_into_windows(all_events)
    if len(windows) < 2:
        print(f"  {uid}: {len(windows)} window(s) – skipping (need >= 2)")
        continue

    graphs: List[SimpleNamespace] = []
    for i, w in enumerate(windows):
        try:
            tg = processor.process(w, uid, f"{uid}_w{i}")
            graphs.append(converter.convert(tg))
        except Exception as exc:
            print(f"  {uid} window {i}: {exc}")

    if len(graphs) >= 2:
        user_graphs[uid] = graphs
        print(f"  ✓ {uid}: {len(graphs)} graphs  (from {len(windows)} windows)")
    else:
        print(f"  {uid}: only {len(graphs)} valid graph(s) – skipping")

print(f"\nUsers ready for triplet training: {list(user_graphs.keys())}")
if not user_graphs:
    raise RuntimeError(
        "No users have sufficient data for triplet training. "
        "Check your data source or increase SYNTH_SESSIONS_PER_USER."
    )


In [ ]:
# ── Triplet Training Loop ────────────────────────────────────────────────────

model    = SiameseGATNetwork()
trainer  = GATTrainer(model)
all_uids = list(user_graphs.keys())

print("=" * 60)
print("Starting triplet training")
print("=" * 60)
print(f"  Device  : {DEVICE}")
print(f"  Users   : {len(all_uids)}")
print(f"  Epochs  : {NUM_EPOCHS}")
print(f"  LR      : {LEARNING_RATE}")
print(f"  Margin  : {TRIPLET_MARGIN}")
print("-" * 60)

t_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    n_batches  = 0

    for uid in all_uids:
        graphs     = user_graphs[uid]
        other_uids = [u for u in all_uids if u != uid]

        for i in range(len(graphs)):
            for j in range(len(graphs)):
                if i == j:
                    continue

                anchor   = graphs[i]
                positive = graphs[j]

                if other_uids:
                    # True negative: a session from a different user
                    neg_uid  = random.choice(other_uids)
                    negative = random.choice(user_graphs[neg_uid])
                else:
                    # Single-user fallback: synthetic negative via Gaussian noise
                    noisy_x  = anchor.x + torch.randn_like(anchor.x) * 0.5
                    negative = SimpleNamespace(
                        x=noisy_x.clamp(0.0, 1.0),
                        edge_index=anchor.edge_index,
                        temporal_features=anchor.temporal_features,
                        batch=anchor.batch,
                    )

                loss = trainer.step(anchor, positive, negative)
                epoch_loss += loss
                n_batches  += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    lr_now   = trainer.opt.param_groups[0]["lr"]

    if epoch == 1 or epoch % 5 == 0 or epoch == NUM_EPOCHS:
        elapsed = time.time() - t_start
        print(
            f"  Epoch {epoch:3d}/{NUM_EPOCHS}"
            f"  loss={avg_loss:.5f}"
            f"  batches={n_batches}"
            f"  lr={lr_now:.2e}"
            f"  elapsed={elapsed:.1f}s"
        )

    trainer.sched.step()

total_time = time.time() - t_start
print("-" * 60)
print(f"✓ Training complete  –  {total_time:.1f}s  ({total_time / NUM_EPOCHS:.2f}s/epoch)")


In [ ]:
# ── Save Checkpoint & Upload to Azure ───────────────────────────────────────

# ── Save locally ─────────────────────────────────────────────────────────────
torch.save(model.state_dict(), CHECKPOINT_PATH)
size_mb = os.path.getsize(CHECKPOINT_PATH) / 1_048_576
print(f"✓ Checkpoint saved locally: {CHECKPOINT_PATH}  ({size_mb:.2f} MiB)")

# ── Upload to Azure Blob Storage ──────────────────────────────────────────────
azure_ok = False
if blob.enabled:
    azure_ok = blob.upload(CHECKPOINT_PATH, CHECKPOINT_BLOB)
    if azure_ok:
        print(f"  Container blobs: {blob.list_blobs()}")
    else:
        print("Azure upload failed – checkpoint retained locally only")
else:
    print("Azure not configured – checkpoint kept locally only")
    print("Set AZURE_STORAGE_CONNECTION_STRING to enable upload")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("Training Summary")
print("=" * 60)
print(f"  Device         : {DEVICE}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Users trained  : {len(all_uids)}")
print(f"  Training time  : {total_time:.1f}s")
print(f"  Checkpoint     : {CHECKPOINT_PATH}  ({size_mb:.2f} MiB)")
azure_status = (
    f"Uploaded to blob '{CHECKPOINT_BLOB}' in container '{AZURE_CONTAINER_NAME}'"
    if azure_ok else
    "Not uploaded (Azure not configured or upload failed)"
)
print(f"  Azure status   : {azure_status}")
